In [ ]:
# 🎯 Example 1: Sequential Multi-Agent System (Pipeline)
# 🧠 Scenario

# “A travel company has different employees (agents):

# Planner → decides steps

# Flight Agent → finds flights

# Weather Agent → checks weather

# Decision Agent → gives final answer”


# AGENT 1: PLANNER

def planner_agent(user_query):
    print("\n[Planner Agent] Creating plan...")
    return ["flight", "weather", "decision"]



# AGENT 2: FLIGHT AGENT

def flight_agent():
    print("\n[Flight Agent] Fetching flights...")
    return [
        {"airline": "IndiGo", "price": 4500},
        {"airline": "Air India", "price": 5200}
    ]



# AGENT 3: WEATHER AGENT

def weather_agent():
    print("\n[Weather Agent] Checking weather...")
    return {"condition": "Clear", "temp": 28}



# AGENT 4: DECISION AGENT

def decision_agent(flights, weather):
    print("\n[Decision Agent] Making decision...")

    cheapest = min(flights, key=lambda x: x["price"])

    if weather["condition"] == "Rain":
        return "Avoid travel due to bad weather"

    return f"Book {cheapest['airline']} at ₹{cheapest['price']}"



# MAIN MULTI-AGENT SYSTEM

def travel_multi_agent(user_query):
    print("User Query:", user_query)

    plan = planner_agent(user_query)

    flights = None
    weather = None

    for step in plan:
        if step == "flight":
            flights = flight_agent()

        elif step == "weather":
            weather = weather_agent()

        elif step == "decision":
            result = decision_agent(flights, weather)

    return result


# RUN
response = travel_multi_agent("Plan my trip Delhi to Mumbai")
print("\nFinal Answer:", response)

User Query: Plan my trip Delhi to Mumbai

[Planner Agent] Creating plan...

[Flight Agent] Fetching flights...

[Weather Agent] Checking weather...

[Decision Agent] Making decision...

Final Answer: Book IndiGo at ₹4500


Scenario
“A hospital uses different employees (agents) to handle patient care in sequence.”

================================
AGENT 1: Intake Agent (Planner)
- Collects patient symptoms and history
- Decides which steps are needed (tests, consultations, etc.)

================================
AGENT 2: Diagnostic Agent
- Orders lab tests or scans
- Interprets results and identifies possible conditions

================================
AGENT 3: Treatment Agent
- Suggests treatment options (medication, therapy, surgery)
- Considers patient preferences and medical guidelines

================================
AGENT 4: Decision Agent
- Reviews all inputs (history, diagnostics, treatment options)
- Provides the final recommendation to the patient

In [ ]:
from huggingface_hub import InferenceClient
from google.colab import userdata

# Load API key
api_key = userdata.get('hugging_api')


client = InferenceClient(
    model="mistralai/Mistral-7B-Instruct-v0.3",
    token=api_key
)


# AGENT 1: INTAKE AGENT

def intake_agent(user_input):
    print("\n[Intake Agent] Collecting patient data...")
    patient_data = {
        "symptoms": user_input,
        "history": "No major past illness"
    }
    plan = ["diagnosis", "treatment", "decision"]
    return patient_data, plan


# HELPER FUNCTION

def call_llm(prompt, max_tokens=200):
    try:
        response = client.chat_completion(
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"[LLM Error] {e}")
        return "Unable to generate response."


# AGENT 2: DIAGNOSTIC AGENT

def diagnostic_agent(patient_data):
    print("\n[Diagnostic Agent] Analyzing symptoms...")
    prompt = f"""You are a medical assistant.
Patient symptoms: {patient_data['symptoms']}
Medical history: {patient_data['history']}
What could be the possible diagnosis? Be concise and clear."""
    return call_llm(prompt, 150)


# AGENT 3: TREATMENT AGENT

def treatment_agent(diagnosis):
    print("\n[Treatment Agent] Suggesting treatment...")
    prompt = f"""You are a medical assistant.
Diagnosis: {diagnosis}
Suggest possible treatments in simple terms. Be concise."""
    return call_llm(prompt, 150)


# AGENT 4: DECISION AGENT

def decision_agent(patient_data, diagnosis, treatment):
    print("\n[Decision Agent] Final recommendation...")
    prompt = f"""You are a senior doctor.
Patient Info: {patient_data}
Diagnosis: {diagnosis}
Treatment Options: {treatment}
Give a final recommendation for the patient in simple terms. Be concise."""
    return call_llm(prompt, 200)


# MAIN MULTI-AGENT SYSTEM

def hospital_multi_agent(user_input):
    print("=" * 50)
    print("Patient Input:", user_input)
    print("=" * 50)

    patient_data, plan = intake_agent(user_input)

    diagnosis = None
    treatment = None
    result    = None

    for step in plan:
        if step == "diagnosis":
            diagnosis = diagnostic_agent(patient_data)
            print("\n>> Diagnosis:\n", diagnosis)

        elif step == "treatment":
            treatment = treatment_agent(diagnosis)
            print("\n>> Treatment Options:\n", treatment)

        elif step == "decision":
            result = decision_agent(patient_data, diagnosis, treatment)

    return result


# RUN

response = hospital_multi_agent("Fever, headache, and body pain for 2 days")
print("\n" + "=" * 50)
print("FINAL RECOMMENDATION:\n", response)
print("=" * 50)

Patient Input: Fever, headache, and body pain for 2 days

[Intake Agent] Collecting patient data...

[Diagnostic Agent] Analyzing symptoms...
[LLM Error] (Request ID: Root=1-69bb961b-564b791647e7ac862d5316a0;f0f72779-efde-4cf7-b6ba-98f869488371)

Bad request:
{'message': "The requested model 'mistralai/Mistral-7B-Instruct-v0.3' is not a chat model.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_supported'}

>> Diagnosis:
 Unable to generate response.

[Treatment Agent] Suggesting treatment...
[LLM Error] (Request ID: Root=1-69bb961b-52cf92ba20d530310835720b;2936e9b2-b8ad-4f0d-a2b9-f1e046885c7f)

Bad request:
{'message': "The requested model 'mistralai/Mistral-7B-Instruct-v0.3' is not a chat model.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_supported'}

>> Treatment Options:
 Unable to generate response.

[Decision Agent] Final recommendation...
[LLM Error] (Request ID: Root=1-69bb961b-28270ef8688db9264e134f73;d967c49c-767d-45da-a7aa

In [ ]:
# Example 2: Manager–Worker Multi-Agent System
# 🧠 Scenario

# “Now instead of fixed flow, we introduce a Manager Agent
# that assigns tasks dynamically to worker


# WORKER AGENTS

def flight_agent():
    print("[Flight Agent] Working...")
    return [{"airline": "IndiGo", "price": 4500},
            {"airline": "Air India", "price": 5200}]


def weather_agent():
    print("[Weather Agent] Working...")
    return {"condition": "Clear", "temp": 28}



# MANAGER AGENT

def manager_agent(user_query):
    print("\n[Manager Agent] Analyzing task...")

    tasks = []

    if "trip" in user_query.lower():
        tasks = ["flight", "weather"]

    return tasks



# EXECUTION

def run_system(user_query):
    print("User Query:", user_query)

    tasks = manager_agent(user_query)

    results = {}

    for task in tasks:
        if task == "flight":
            results["flights"] = flight_agent()

        elif task == "weather":
            results["weather"] = weather_agent()

    # Final decision
    cheapest = min(results["flights"], key=lambda x: x["price"])

    return f"Manager Decision: Book {cheapest['airline']} at ₹{cheapest['price']}"


# RUN
response = run_system("Plan my trip")
print("\nFinal Answer:", response)


User Query: Plan my trip

[Manager Agent] Analyzing task...
[Flight Agent] Working...
[Weather Agent] Working...

Final Answer: Manager Decision: Book IndiGo at ₹4500


In [ ]:
# “Here, the manager decides:

# what tasks are needed

# which agents to call

# This is called a hierarchical multi-agent system.”

Scenario
“A hospital uses different employees (agents) to handle patient care in sequence.”

================================
AGENT 1: Intake Agent (Planner)
- Collects patient symptoms and history
- Decides which steps are needed (tests, consultations, etc.)

================================
AGENT 2: Diagnostic Agent
- Orders lab tests or scans
- Interprets results and identifies possible conditions

================================
AGENT 3: Treatment Agent
- Suggests treatment options (medication, therapy, surgery)
- Considers patient preferences and medical guidelines

================================
AGENT 4: Decision Agent
- Reviews all inputs (history, diagnostics, treatment options)
- Provides the final recommendation to the patientx

In [ ]:
from google import genai
from google.genai import types
from google.colab import userdata

# 1. Initialize the Google GenAI Client
# Ensure your 'api_key' is set in Colab Secrets
api_key = userdata.get('api_key')
client = genai.Client(api_key=api_key)

# The specific model ID for the 3.1 Flash Lite preview
MODEL_ID = "gemini-3.1-flash-lite-preview"


# HELPER FUNCTION (TEXT GENERATION)

def call_llm(system_instruction, user_prompt, max_tokens=200):
    try:

        response = client.models.generate_content(
            model=MODEL_ID,
            contents=user_prompt,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                max_output_tokens=max_tokens,
                thinking_config=types.ThinkingConfig(
                    thinking_level=types.ThinkingLevel.LOW
                )
            )
        )
        return response.text
    except Exception as e:
        print(f"[Gemini Error] {e}")
        return "Unable to generate response."


# AGENTS


def intake_agent(user_input):
    print("\n[Intake Agent] Collecting patient data...")

    patient_data = {"symptoms": user_input, "history": "No major past illness"}
    plan = ["diagnosis", "treatment", "decision"]
    return patient_data, plan

def diagnostic_agent(patient_data):
    print("\n[Diagnostic Agent] Analyzing symptoms...")
    sys_msg = "You are a medical assistant. Provide concise possible diagnoses."
    prompt = f"Symptoms: {patient_data['symptoms']}\nHistory: {patient_data['history']}"
    return call_llm(sys_msg, prompt, 150)

def treatment_agent(diagnosis):
    print("\n[Treatment Agent] Suggesting treatment...")
    sys_msg = "You are a medical assistant. Suggest simple treatments for the given diagnosis."
    prompt = f"Diagnosis: {diagnosis}"
    return call_llm(sys_msg, prompt, 150)

def decision_agent(patient_data, diagnosis, treatment):
    print("\n[Decision Agent] Final recommendation...")
    sys_msg = "You are a senior doctor. Give a final recommendation in simple terms."
    prompt = f"Patient Info: {patient_data}\nDiagnosis: {diagnosis}\nTreatments: {treatment}"
    return call_llm(sys_msg, prompt, 200)


# MAIN SYSTEM

def hospital_multi_agent(user_input):
    print("=" * 50)
    print("Patient Input:", user_input)
    print("=" * 50)

    patient_data, plan = intake_agent(user_input)
    diagnosis, treatment, result = None, None, None

    for step in plan:
        if step == "diagnosis":
            diagnosis = diagnostic_agent(patient_data)
            print("\n>> Diagnosis:\n", diagnosis)
        elif step == "treatment":
            treatment = treatment_agent(diagnosis)
            print("\n>> Treatment Options:\n", treatment)
        elif step == "decision":
            result = decision_agent(patient_data, diagnosis, treatment)

    return result

# RUN
response = hospital_multi_agent("Fever, headache, and body pain for 2 days")
print("\n" + "=" * 50)
print("FINAL RECOMMENDATION:\n", response)
print("=" * 50)

Patient Input: Fever, headache, and body pain for 2 days

[Intake Agent] Collecting patient data...

[Diagnostic Agent] Analyzing symptoms...

>> Diagnosis:
 The symptoms of fever, headache, and body pain (myalgia) are non-specific and common to many acute conditions.

[Treatment Agent] Suggesting treatment...

>> Treatment Options:
 Since the symptoms of fever, headache, and body aches are non-specific, the goal of treatment is generally **supportive care** to help your body recover while

[Decision Agent] Final recommendation...

FINAL RECOMMENDATION:
 Based on the symptoms you've described, you are likely dealing with a common viral illness. Since you have no other major health issues, the best course of action is to support your body while it fights off the infection.

Here is my recommendation:

1.  **Rest:** Your body needs extra energy to recover. Prioritize sleeping and avoid strenuous activity.
2.  **Stay


Scenario: Corporate Market Research & Strategy
A company wants to explore launching a new product in a competitive market. The Manager Agent oversees the process and delegates tasks to specialized workers.

👩‍💼 Manager Agent
- Receives the overall goal: “Evaluate feasibility of launching Product Y in Asia.”
- Dynamically assigns tasks to worker agents depending on what’s needed.
- Example: If budget is unclear → send to Finance Worker. If regulations are complex → send to Legal Worker.

================================
📊 Worker Agents
================================
- Market Research Worker
- Collects competitor data, customer preferences, and demand forecasts.
- Reports: “High demand in Tier‑1 cities, moderate competition.”
- Finance Worker
- Analyzes budget, ROI, and pricing strategy.
- Reports: “Estimated $10M investment, ROI in 2 years.”
- Operations Worker
- Evaluates supply chain, production capacity, and logistics.
- Reports: “Factories can scale up, but shipping costs are high.”
- Legal Worker
- Reviews compliance, intellectual property, and regional regulations.
- Reports: “Trademark available, but import laws require certification.”
- HR Worker
- Assesses staffing needs and training requirements.
- Reports: “Need 50 new hires for customer support and sales.”

In [ ]:
# Install the latest SDK
!pip install -U google-genai

import time
from google import genai
from google.genai import types
from google.colab import userdata


try:
    api_key = userdata.get('api_key')
    client = genai.Client(api_key=api_key)
except Exception as e:
    print(f"Error loading API Key: {e}")


MODEL_ID = "gemini-3.1-flash-lite-preview"


# HELPER FUNCTION (ROBUST LLM CALL)

def call_llm(system_instruction, user_prompt, max_tokens=300):
    """Safe wrapper for Gemini API calls with error handling."""
    try:
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=user_prompt,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                max_output_tokens=max_tokens,

                safety_settings=[
                    types.SafetySetting(category="HARM_CATEGORY_HARASSMENT", threshold="BLOCK_NONE"),
                    types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH", threshold="BLOCK_NONE"),
                    types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="BLOCK_NONE"),
                    types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="BLOCK_NONE"),
                ],
                thinking_config=types.ThinkingConfig(thinking_level="minimal")
            )
        )

        if response and response.text:
            return response.text.strip()
        return ""
    except Exception as e:
        print(f"[Gemini Error] {e}")
        return ""


# WORKER AGENTS


def market_research_worker(goal):
    print("[Worker] Analyzing Market Research...")
    sys = "You are a Market Research Analyst. Provide a concise 3-4 line report."
    return call_llm(sys, f"Goal: {goal}")

def finance_worker(goal):
    print("[Worker] Analyzing Financials...")
    sys = "You are a Financial Analyst. Provide a concise 3-4 line report."
    return call_llm(sys, f"Goal: {goal}")

def operations_worker(goal):
    print("[Worker] Evaluating Operations...")
    sys = "You are an Operations Manager. Provide a concise 3-4 line report."
    return call_llm(sys, f"Goal: {goal}")

def legal_worker(goal):
    print("[Worker] Reviewing Legal Aspects...")
    sys = "You are a Legal Compliance Officer. Provide a concise 3-4 line report."
    return call_llm(sys, f"Goal: {goal}")

def hr_worker(goal):
    print("[Worker] Assessing Staffing (HR)...")
    sys = "You are an HR Manager. Provide a concise 3-4 line report."
    return call_llm(sys, f"Goal: {goal}")


# MANAGER AGENT

def manager_agent(goal):
    print("\n[Manager Agent] Delegating tasks...")

    planning_sys = "You are a Strategic Manager. Decide which departments are needed for the goal."
    planning_prompt = (
        f"Goal: {goal}\n"
        "Departments: Market Research, Finance, Operations, Legal, HR\n"
        "Return ONLY a comma-separated list of the relevant departments. "
        "Example: Market Research, Finance, Legal"
    )


    departments_raw = call_llm(planning_sys, planning_prompt, 100)


    if not departments_raw:
        print("[Manager Agent] No departments returned. Defaulting to all.")
        departments_raw = "Market Research, Finance, Operations, Legal, HR"

    print(f"[Manager Agent] Departments selected: {departments_raw}")


    selected_list = [d.strip().lower() for d in departments_raw.replace('.', '').split(",")]
    reports = {}


    if "market" in str(selected_list): reports["Market Research"] = market_research_worker(goal)
    if "finance" in str(selected_list): reports["Finance"] = finance_worker(goal)
    if "operation" in str(selected_list): reports["Operations"] = operations_worker(goal)
    if "legal" in str(selected_list): reports["Legal"] = legal_worker(goal)
    if "hr" in str(selected_list) or "human" in str(selected_list): reports["HR"] = hr_worker(goal)


    print("\n[Manager Agent] Synthesizing final strategy...")
    all_reports = "\n".join([f"{dept} Report: {report}" for dept, report in reports.items()])

    final_sys = "You are a Senior Executive. Provide a high-level strategic recommendation (5-6 lines)."
    final_prompt = f"Original Goal: {goal}\n\nReports:\n{all_reports}\n\nRecommendation, Risks, and Next Steps:"

    final_recommendation = call_llm(final_sys, final_prompt, 400)
    return reports, final_recommendation


# RUN SYSTEM

def run_corporate_strategy(goal):
    print("=" * 60)
    print(f"STRATEGY SYSTEM: {goal}")
    print("=" * 60)

    reports, final_rec = manager_agent(goal)

    print("\n" + "=" * 60)
    print("FINAL STRATEGIC RECOMMENDATION")
    print("=" * 60)
    print(final_rec if final_rec else "Strategy generation failed.")
    print("=" * 60)

# Run the process
run_corporate_strategy("Evaluate feasibility of launching Product Y in Asia.")

STRATEGY SYSTEM: Evaluate feasibility of launching Product Y in Asia.

[Manager Agent] Delegating tasks...
[Manager Agent] Departments selected: Market Research, Finance, Operations, Legal, HR
[Worker] Analyzing Market Research...
[Worker] Analyzing Financials...
[Worker] Evaluating Operations...
[Worker] Reviewing Legal Aspects...
[Worker] Assessing Staffing (HR)...

[Manager Agent] Synthesizing final strategy...

FINAL STRATEGIC RECOMMENDATION
**Recommendation:** I approve a phased, low-risk market entry for Product Y, starting with a targeted pilot in Singapore to validate operational, regulatory, and cultural alignment. 

**Risks:** We face significant exposure regarding fragmented cross-border data privacy laws, complex supply chain logistics, and aggressive local competition. 

**Next Steps:** We will initiate a Q3 pilot program, tasking Legal and Operations with building a region-specific compliance framework and localized supply chain architecture. Simultaneously, we must prior

Agent 1: Crisis Coordinator (Broadcaster)
- Broadcasts: “Data breach detected in customer database. Immediate response required.”
- Sends this to all other agents simultaneously.

🛡️ Agent 2: IT Security Agent
- Receives broadcast.
- Responds: “Isolate affected servers, patch vulnerabilities, start forensic analysis.”

📞 Agent 3: Communications Agent
- Receives broadcast.
- Responds: “Draft internal memo, prepare press release, notify stakeholders.”

💰 Agent 4: Finance Agent
- Receives broadcast.
- Responds: “Estimate financial impact, allocate emergency funds, review insurance coverage.”

👩‍⚖️ Agent 5: Legal Agent
- Receives broadcast.
- Responds: “Assess regulatory obligations, prepare compliance reports, advise on liability.”

👩‍💼 Agent 6: HR Agent
- Receives broadcast.
- Responds: “Brief employees, provide guidance on handling customer queries, ensure morale support.”

🧑‍⚖️ Agent 7: Decision Agent (Coordinator)
- Collects all responses.
- Integrates into a final crisis response plan:
“Servers isolated, communications prepared, financial impact assessed, compliance secured, employees briefed.”

In [ ]:
# Install the SDK if you haven't already
!pip install -U google-genai

from google import genai
from google.genai import types
from google.colab import userdata


api_key = userdata.get('api_key')
client = genai.Client(api_key=api_key)

MODEL_ID = "gemini-3.1-flash-lite-preview"


# HELPER FUNCTION (ROBUST LLM CALL)

def call_llm(system_instruction, user_prompt, max_tokens=300):
    try:
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=user_prompt,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                max_output_tokens=max_tokens,

                safety_settings=[
                    types.SafetySetting(category="HARM_CATEGORY_HARASSMENT", threshold="BLOCK_NONE"),
                    types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH", threshold="BLOCK_NONE"),
                    types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="BLOCK_NONE"),
                    types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="BLOCK_NONE"),
                ],
                thinking_config=types.ThinkingConfig(thinking_level="minimal")
            )
        )

        return response.text.strip() if response and response.text else "No response generated."
    except Exception as e:
        print(f"[Gemini Error] {e}")
        return "Service unavailable."


# AGENTS


def crisis_coordinator(crisis):
    print("\n[Crisis Coordinator] Broadcasting alert...")
    broadcast = f"CRISIS ALERT: {crisis}. Immediate response required."
    return broadcast

def it_security_agent(broadcast):
    print("[Agent] IT Security responding...")
    sys = "You are an IT Security Expert. List 3 immediate security actions."
    return call_llm(sys, broadcast)

def communications_agent(broadcast):
    print("[Agent] Communications responding...")
    sys = "You are a Corp Comm Manager. List 3 immediate communication steps."
    return call_llm(sys, broadcast)

def finance_agent(broadcast):
    print("[Agent] Finance responding...")
    sys = "You are a Financial Risk Manager. List 3 immediate financial actions."
    return call_llm(sys, broadcast)

def legal_agent(broadcast):
    print("[Agent] Legal responding...")
    sys = "You are a Legal Compliance Officer. List 3 immediate legal steps."
    return call_llm(sys, broadcast)

def hr_agent(broadcast):
    print("[Agent] HR responding...")
    sys = "You are an HR Manager. List 3 immediate HR/staffing actions."
    return call_llm(sys, broadcast)


# DECISION & MAIN SYSTEM


def decision_agent(crisis, responses):
    print("\n[Decision Agent] Synthesizing final plan...")
    all_responses = "\n".join([f"{dept}: {resp}" for dept, resp in responses.items()])

    sys = "You are a Crisis Decision Coordinator. Integrate departmental inputs into one unified plan."
    prompt = f"Original Crisis: {crisis}\n\nDepartmental Inputs:\n{all_responses}\n\nWrite a 6-8 line final plan."

    return call_llm(sys, prompt, 400)

def run_crisis_management(crisis):
    print("=" * 60)
    print("CRISIS MANAGEMENT SYSTEM ACTIVE")
    print("=" * 60)

    broadcast = crisis_coordinator(crisis)


    responses = {
        "IT Security": it_security_agent(broadcast),
        "Communications": communications_agent(broadcast),
        "Finance": finance_agent(broadcast),
        "Legal": legal_agent(broadcast),
        "HR": hr_agent(broadcast)
    }


    for dept, resp in responses.items():
        print(f"\n📋 {dept}:\n{resp}")


    final_plan = decision_agent(crisis, responses)

    print("\n" + "=" * 60)
    print("FINAL CRISIS RESPONSE PLAN")
    print("=" * 60)
    print(final_plan)
    print("=" * 60)

# RUN THE SYSTEM
run_crisis_management("Data breach detected in customer database.")

CRISIS MANAGEMENT SYSTEM ACTIVE

[Crisis Coordinator] Broadcasting alert...
[Agent] IT Security responding...
[Agent] Communications responding...
[Agent] Finance responding...
[Agent] Legal responding...
[Agent] HR responding...

📋 IT Security:
As an IT Security Expert, you must act immediately to contain the breach and preserve evidence. Execute these three actions now:

### 1. Isolate Affected Systems
**Action:** Disconnect the compromised database servers and associated application servers from the network.
*   **Why:** You must stop the attacker from exfiltrating further data or moving laterally through your infrastructure. 
*   **Important:** Do **not** shut down the servers (powering off wipes volatile RAM, which contains critical forensic evidence like active connections and encryption keys). Instead, use network isolation (VLAN isolation or firewall rules) to cut off external access while keeping the machines running for investigation.

### 2. Force Credential Resets and Revok

Scenario: Corporate Product Launch Broadcast
Imagine a company preparing to launch Product X in Q3. The Coordinator Agent (like a corporate program manager) sends out a broadcast announcement to all departments at once:
“Product X launch in Q3, target market North America, budget $5M.”


📢 Coordinator Agent (Broadcaster)
- Sends the launch announcement to all departments simultaneously.
- This is implemented in the code by the broadcast() function, which uses asyncio.gather() to run all agents in parallel.

📈 Marketing Agent
- Receives the broadcast.
- Responds with a marketing strategy: campaigns, channels, and positioning.
💰 Finance Agent
- Receives the broadcast.
- Responds with budget allocation and ROI forecasts.
🏭 Operations Agent
- Receives the broadcast.
- Responds with production and supply chain actions.
👩‍⚖️ Legal Agent
- Receives the broadcast.
- Responds with compliance checks and contract actions.
👩‍💼 HR Agent
- Receives the broadcast.
- Responds with staffing and training plans.

🧑‍⚖️ Decision Agent
- Collects all responses.
- Integrates them into a Final Corporate Launch Plan.
- In the code, this is the decision_agent() function that merges all outputs into one consolidated plan.

In [ ]:
import asyncio

async def marketing_agent(message):
    await asyncio.sleep(1)
    return "Marketing: Digital campaigns, social media ads, NA-focused branding"

async def finance_agent(message):
    await asyncio.sleep(1)
    return "Finance: Allocate $5M budget, forecast 20% ROI, monitor CAC"

async def operations_agent(message):
    await asyncio.sleep(1)
    return "Operations: Scale production, secure supply chain, optimize logistics"

async def legal_agent(message):
    await asyncio.sleep(1)
    return "Legal: Ensure NA compliance, review contracts, IP protection"

async def hr_agent(message):
    await asyncio.sleep(1)
    return "HR: Hire regional team, conduct training, align workforce"


async def broadcast(message):
    print("\n [Coordinator] Broadcasting message to all agents...\n")
    print(f"Message: {message}\n")

    results = await asyncio.gather(
        marketing_agent(message),
        finance_agent(message),
        operations_agent(message),
        legal_agent(message),
        hr_agent(message)
    )

    return results


def decision_agent(responses):
    print("\n [Decision Agent] Synthesizing responses...\n")

    final_plan = "\n".join(responses)

    final_summary = f"""
 Final Corporate Launch Plan:

{final_plan}

- Marketing strategy ready
- Budget allocated and ROI estimated
- Operations prepared for scale
- Legal compliance ensured
- HR aligned for execution

 Recommendation: Proceed with Product X launch in Q3.
"""

    return final_summary


async def main():
    message = "Product X launch in Q3, target market North America, budget $5M."

    responses = await broadcast(message)
    final_output = decision_agent(responses)

    print(final_output)



await main()


 [Coordinator] Broadcasting message to all agents...

Message: Product X launch in Q3, target market North America, budget $5M.


 [Decision Agent] Synthesizing responses...


 Final Corporate Launch Plan:

Marketing: Digital campaigns, social media ads, NA-focused branding
Finance: Allocate $5M budget, forecast 20% ROI, monitor CAC
Operations: Scale production, secure supply chain, optimize logistics
Legal: Ensure NA compliance, review contracts, IP protection
HR: Hire regional team, conduct training, align workforce

- Marketing strategy ready
- Budget allocated and ROI estimated
- Operations prepared for scale
- Legal compliance ensured
- HR aligned for execution

 Recommendation: Proceed with Product X launch in Q3.



In [ ]:



import asyncio
import time
from google import genai
from google.genai import types
from google.colab import userdata


api_key = userdata.get('api_key')
client = genai.Client(api_key=api_key)


MODEL_ID = "gemini-3.1-flash-lite-preview"


async def call_llm(system_instruction, user_prompt, max_tokens=1000, retries=3):
    """Handles retries, rate limits, and safety filters."""
    for attempt in range(retries):
        try:

            response = await asyncio.to_thread(
                client.models.generate_content,
                model=MODEL_ID,
                contents=user_prompt,
                config=types.GenerateContentConfig(
                    system_instruction=system_instruction,
                    max_output_tokens=max_tokens,
                    temperature=0.7,

                    safety_settings=[
                        types.SafetySetting(category="HARM_CATEGORY_HARASSMENT", threshold="BLOCK_NONE"),
                        types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH", threshold="BLOCK_NONE"),
                        types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="BLOCK_NONE"),
                        types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="BLOCK_NONE"),
                    ],
                    thinking_config=types.ThinkingConfig(thinking_level="minimal")
                )
            )

            if response and response.text:
                return response.text.strip()
            return "Error: Empty response from model."

        except Exception as e:
            if "429" in str(e) or "quota" in str(e).lower():
                wait_time = (attempt + 1) * 5
                print(f"  [Rate Limited] Retrying in {wait_time}s...")
                await asyncio.sleep(wait_time)
            else:
                return f"[LLM Error] {e}"

    return "[LLM Error] Max retries exhausted."


# 🧠 SHARED MEMORY

class SharedMemory:
    def __init__(self):
        self.data = {}

    def update(self, key, value):
        print(f"  [Memory Updated] {key} updated.")
        self.data[key] = value

    def get(self, key):
        return self.data.get(key, "No data available.")

# ================================
# 🤖 AGENTS (Refactored for System Instructions)
# ================================

async def search_agent(goal, memory):
    print("\n[Search Agent] Gathering market facts...")
    sys = "You are a research agent. Extract key facts and concise bullet points."
    result = await call_llm(sys, f"Find key facts about: {goal}")
    memory.update("search_data", result)

async def analyst_agent(goal, memory):
    print("[Analyst Agent] Extracting insights...")
    data = memory.get("search_data")
    sys = "You are a senior analyst. Extract trends and opportunities from research data."
    result = await call_llm(sys, data)
    memory.update("analysis_data", result)

async def writer_agent(goal, memory):
    print("[Writer Agent] Drafting formal report...")
    analysis = memory.get("analysis_data")
    sys = "You are a professional business writer. Create a well-structured report with headers."
    result = await call_llm(sys, f"Base this report on the following analysis: {analysis}")
    memory.update("report_draft", result)

async def qa_agent(goal, memory):
    print("[QA Agent] Finalizing and polishing...")
    draft = memory.get("report_draft")
    sys = "You are a QA editor. Polished the report for clarity. Return only the final text."
    result = await call_llm(sys, draft)
    memory.update("final_report", result)

# ================================
# 🧑‍💼 ORCHESTRATOR
# ================================
async def orchestrator(goal):
    print("=" * 60)
    print(f"🚀 STARTING PIPELINE: {goal}")
    print("=" * 60)

    memory = SharedMemory()

    # Sequential execution (Wait for each stage to finish)
    await search_agent(goal, memory)
    await analyst_agent(goal, memory)
    await writer_agent(goal, memory)
    await qa_agent(goal, memory)

    print("\n" + "=" * 60)
    print("FINAL MARKET REPORT")
    print("=" * 60)
    print(memory.get("final_report"))
    print("=" * 60)

# ================================
# ▶️ RUN
# ================================
goal = "Comprehensive EV market report for India in 2025"
await orchestrator(goal)

🚀 STARTING PIPELINE: Comprehensive EV market report for India in 2025

[Search Agent] Gathering market facts...
  [Memory Updated] search_data updated.
[Analyst Agent] Extracting insights...
  [Memory Updated] analysis_data updated.
[Writer Agent] Drafting formal report...
  [Memory Updated] report_draft updated.
[QA Agent] Finalizing and polishing...
  [Memory Updated] final_report updated.

FINAL MARKET REPORT
# Strategic Outlook: The Indian EV Landscape 2025

**To:** Strategic Stakeholders and Investment Committees  
**From:** Senior Analyst  
**Date:** May 22, 2024  
**Subject:** Market Analysis and Strategic Roadmap for the Indian Electric Vehicle Sector

---

### 1. Executive Summary: The Tipping Point
The Indian electric vehicle (EV) sector is shifting from a government-subsidized early-adoption phase to a market-driven mass-adoption phase. With an anticipated market valuation of $15B–$20B, the sector is becoming a cornerstone of the domestic economy. Strategic success in 2025 w